# 03 - r3 chemistry RGCN, symmetric protocol (A100)

The 16-config grid at seeds 0/1 under the unified HGT budget (200/15, weight
decay 1e-4), winner at 10 seeds under both budgets, then the corrected
eq. (2) protocol against M5'.

**Device homogeneity rule:** the bundle carries 17 grid configs finished on
CPU. A mixed CPU/GPU grid is exactly the asymmetry this paper audits, so the
first cell moves those CPU partials to `cpu_superseded/` (kept as audit
trail, marked `superseded_by_gpu`) and the whole grid re-runs on the A100.
The verdict uses GPU runs only.

In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/who-inherits")  # for colab_common if bundle not yet unzipped
try:
    import colab_common as cc
except ImportError:
    # colab_common ships inside the bundle; bootstrap: mount, unzip, import
    from google.colab import drive as _d; _d.mount("/content/drive")
    import subprocess
    subprocess.run(["unzip", "-q", "-o",
                    "/content/drive/MyDrive/who-inherits/colab_bundle.zip",
                    "-d", "/content/work"], check=True)
    sys.path.insert(0, "/content/work/colab")
    import colab_common as cc
else:
    cc.mount_drive()
sys.path.insert(0, "/content/work/colab")
import colab_common as cc
cc.setup_workspace()
cc.verify_frozen_hashes()
print(cc.run_meta())

In [ ]:
import torch
assert torch.cuda.is_available(), "select an A100 runtime"
print(torch.cuda.get_device_name(0))
# determinism: seeds are set per run by the script; cudnn flags here.
# PyG scatter/segment kernels have no deterministic GPU implementation for
# every op, so warn_only=True and the fact is RECORDED, not hidden.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
import json
from pathlib import Path
det_note = {"cudnn_deterministic": True, "cudnn_benchmark": False,
            "use_deterministic_algorithms": "True (warn_only=True)",
            "note": "PyG hetero scatter ops fall back to nondeterministic "
                    "CUDA kernels; per-seed results on GPU may vary at ~1e-3 "
                    "AUC-PR across reruns. Recorded rather than pretended away.",
            **cc.run_meta()}
outdir = Path("/content/work/results/robustness/rgcn_symmetric")
outdir.mkdir(parents=True, exist_ok=True)
(outdir / "DETERMINISM_NOTE.json").write_text(json.dumps(det_note, indent=2))

In [ ]:
# supersede the CPU partials (kept, marked, excluded from the GPU grid)
import json, shutil
from pathlib import Path
outdir = Path("/content/work/results/robustness/rgcn_symmetric")
sup = outdir / "cpu_superseded"
sup.mkdir(exist_ok=True)
moved = []
for p in sorted(outdir.glob("rgcn_sym_*.json")):
    d = json.loads(p.read_text())
    if d.get("device") != "cuda":
        d["superseded_by_gpu"] = True
        (sup / p.name).write_text(json.dumps(d, indent=2))
        p.unlink()
        moved.append(p.name)
print(f"moved {len(moved)} CPU partials to cpu_superseded/")
cc.sync_to_drive()

In [ ]:
ckpt = cc.start_checkpoint_thread()
cc.run_script("code/r3_rgcn_symmetric.py",
              env_extra={"DATASET": "chemistry",
                         "DATASET_PATH": "data/clean_dataset_chemistry.parquet"},
              args=("train",))   # skips per-config JSONs already on GPU
cc.stop_checkpoint_thread()
cc.sync_to_drive()

In [ ]:
cc.run_script("code/r3_rgcn_symmetric.py",
              env_extra={"DATASET": "chemistry",
                         "DATASET_PATH": "data/clean_dataset_chemistry.parquet"},
              args=("aggregate",))
import json
v = json.load(open("/content/work/results/robustness/rgcn_symmetric_verdict.json"))
print("grid winner:", v["grid_selection"]["winner"])
print("winner 10-seed:", v["winner_10seed"])
print("rgcn_symmetric exceeds_fair:", v["rgcn_symmetric_exceeds_fair"])
print(json.dumps(v["models"]["rgcn_symmetric"], indent=2))

In [ ]:
cc.write_done_flag("rgcn")